# PMCortex Retrieval Dataset Pipeline

This notebook builds a biomedical document-retrieval benchmark from PMC citation contexts. Papers cited in a query sentence are positive documents; other references from the same source article form topically close hard-negative candidates.

Run the sections in order. Data products are written below `data/pmc_context_queries_v1/` and are intentionally not tracked by Git.


## 1. Discover seed articles with MeSH

Extract coarse MeSH categories from the local descriptor file and use their names as PMC search queries. The resulting PMCID list defines the source-article collection.


In [1]:
import pandas as pd
import tqdm

from pmcortex.mesh import extract_mesh_coarsest_descriptor_categories
from pmcortex.pmcsearch import PMCSearch

# define a set of MeSH categories to use as search queries, based on the coarsest descriptor categories in the MeSH hierarchy
cats = extract_mesh_coarsest_descriptor_categories("../data/desc2026.xml", level=2)
use_keywords = {
    c["tree_number"]: c["name"]
    for c in cats
    if c["tree_number"][0] in ['A', 'B', 'C', 'D', 'E', 'G']
}

# use the MeSH categories as search queries to get seed PMC articles
seed_pmcs = []
with PMCSearch(
    user_agent="PMCortex/1.0 (contact: lorenz@hexemer.eu)",
    max_requests_per_second=2.0,
    max_attempts=3,
    retry_backoff_s=1.0,
) as search:
    for mesh, query in tqdm.tqdm(use_keywords.items()):
        df_ = search.search(query=f"({query}) AND open access[filter]", max_results=5)
        df_ = df_.assign(query=query, mesh=mesh)
        seed_pmcs.append(df_)

# combine and filter the seed PMC articles
seed_pmcs = pd.concat(seed_pmcs)
filtered_seeds = (
    seed_pmcs
    .query('not title.str.contains("correction", case=False)')
    .query('pmcid.str.startswith("PMC")')
    .groupby(['pmcid', 'title'])
    .agg(list)
    .reset_index()
)
filtered_seeds.to_csv("../data/mesh_based_pmc_seed_openaccess_small.csv", index=False)

/home/lorenz/Environments/PMCortex/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1601/1601 [29:19<00:00,  1.10s/it]


## 2. Download and parse PMC articles

Stream seed articles from PMC into a bounded pool of JATS parser processes. Per-article outputs and status records are written atomically, so interrupted runs can resume safely. Successfully parsed JATS files are deleted only after all durable outputs exist; failed inputs are retained for diagnosis.


In [2]:
from pathlib import Path
import pandas as pd

from pmcortex import PMCIngestionPipeline
from pmcortex.downloader import PMCJATSDownloader

DATASET_ROOT = Path("../data/pmc_context_queries_v2")
filtered_seeds = pd.read_csv("../data/mesh_based_pmc_seed_openaccess_small.csv")
required_seeds = filtered_seeds['pmcid'].unique()

In [3]:
required_seeds.shape

(7592,)

In [4]:
with PMCJATSDownloader(
    out_dir=DATASET_ROOT / "xml_jats",
    user_agent="PMCortex/0.2 (contact: lorenz@hexemer.eu)",
    min_interval_s=0.6,
    max_retries=3,
) as downloader:
    ingestion = PMCIngestionPipeline(
        DATASET_ROOT,
        downloader,
        parser_workers=4,
        delete_jats_after_success=True,
        parser_schema_version="citation-normalization-v3-no-spoilers",
        verbose=False,
    )
    processing_records = ingestion.run(required_seeds)
    tables = ingestion.materialize_tables()

PMC ingestion:  29%|██▉       | 2226/7592 [22:18<53:47,  1.66article/s]  2026-08-20 00:05:28.118 | ERROR    | pmcortex.downloader:fetch_oai_xml:264 - Error fetching PMC12994162 (attempt 1): HTTPStatusError("Client error '400 Bad Request' for url 'https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:12994162&metadataPrefix=pmc'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400")
2026-08-20 00:05:29.057 | ERROR    | pmcortex.downloader:fetch_oai_xml:264 - Error fetching PMC12994162 (attempt 2): HTTPStatusError("Client error '400 Bad Request' for url 'https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:12994162&metadataPrefix=pmc'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400")
2026-08-20 00:05:30.833 | ERROR    | pmcortex.downloader:fetch_oai_xml:264 - Error fetching PMC12994162 (attempt 3): HTTPStatusError("Client erro

### Processing audit

Inspect the persistent outcome for every requested article. The `reason` column distinguishes missing records, unavailable full text, download failures, parse failures, and persistence failures. The same records remain available as JSON files under `status/`.


In [5]:
from dataclasses import asdict

status_df = pd.DataFrame(asdict(record) for record in processing_records)
display(status_df.groupby('status').size().rename('article_count'))
display(
    status_df.loc[
        status_df['status'] != 'complete',
        ['pmcid', 'status', 'reason', 'jats_path'],
    ]
)

status
complete          7530
download_error      62
Name: article_count, dtype: int64

,pmcid,status,reason,jats_path
2233,PMC12994162,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
2731,PMC2131870,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
2739,PMC2136121,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
2749,PMC2138895,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
2758,PMC2142833,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
...,...,...,...,...
3219,PMC3464178,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
3228,PMC3471918,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
3238,PMC3487917,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN
3246,PMC3505799,download_error,RuntimeError('Failed to fetch OAI record for P...,NaN


### Reference-count statistics

Summarize how many references were extracted per parsed article. `bibliography_entries` counts all extracted bibliography entries, while `unique_pmid_references` counts distinct references with a PMID and therefore matches the candidate count used by the benchmark filters below. Articles without extracted references are included.


In [7]:
article_ids = (
    pd.read_csv(
        tables.metadata,
        usecols=['pmcid'],
        dtype={'pmcid': 'string'},
    )['pmcid']
    .dropna()
    .drop_duplicates()
)
source_references = pd.read_csv(
    tables.sources,
    usecols=['source_pmcid', 'pmid'],
    dtype={'source_pmcid': 'string', 'pmid': 'string'},
)

bibliography_counts = source_references.groupby('source_pmcid').size()
unique_pmid_counts = (
    source_references.dropna(subset=['pmid'])
    .drop_duplicates(subset=['source_pmcid', 'pmid'])
    .groupby('source_pmcid')
    .size()
)
article_reference_counts = pd.DataFrame({'source_pmcid': article_ids})
article_reference_counts['bibliography_entries'] = (
    article_reference_counts['source_pmcid']
    .map(bibliography_counts)
    .fillna(0)
    .astype('int64')
)
article_reference_counts['unique_pmid_references'] = (
    article_reference_counts['source_pmcid']
    .map(unique_pmid_counts)
    .fillna(0)
    .astype('int64')
)

reference_summary = (
    article_reference_counts[
        ['bibliography_entries', 'unique_pmid_references']
    ]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .loc[['count', 'min', '25%', '50%', 'mean', '75%', '90%', 'max']]
    .rename(index={'count': 'articles', '50%': 'median'})
    .round(1)
)
display(reference_summary)

reference_ranges = pd.cut(
    article_reference_counts['unique_pmid_references'],
    bins=[-1, 0, 9, 19, 24, 49, 99, 199, float('inf')],
    labels=['0', '1–9', '10–19', '20–24', '25–49', '50–99', '100–199', '200+'],
)
reference_distribution = (
    reference_ranges.value_counts(sort=False)
    .rename('article_count')
    .to_frame()
)
reference_distribution['article_share_percent'] = (
    100 * reference_distribution['article_count'] / len(article_reference_counts)
).round(1)
display(reference_distribution)

eligible_article_count = (
    article_reference_counts['unique_pmid_references'] >= 25
).sum()
eligible_article_share = 100 * eligible_article_count / len(article_reference_counts)
print(
    f'Articles with at least 25 distinct PMID references: '
    f'{eligible_article_count:,} ({eligible_article_share:.1f}%)'
)

,bibliography_entries,unique_pmid_references
articles,7530.0,7530.0
min,0.0,0.0
25%,28.0,20.0
median,43.0,34.0
mean,55.6,45.1
75%,66.0,54.0
90%,104.0,89.0
max,1065.0,1064.0


,article_count,article_share_percent
unique_pmid_references,,
0,94,1.2
1–9,539,7.2
10–19,1168,15.5
20–24,716,9.5
25–49,2787,37.0
50–99,1634,21.7
100–199,480,6.4
200+,112,1.5


Articles with at least 25 distinct PMID references: 5,013 (66.6%)


## 3. Apply deterministic query filters

Remove clear formatting problems, explicit citation spoilers, self-citations, and queries with insufficient reference candidates. The self-contained checks are deliberately conservative; ambiguous cross-sentence anaphora is handled later by the Coreference model.


In [8]:
from pmcortex.query_filters import (
    QueryFilterPipeline,
    assert_no_residual_citation_spoilers,
)

pipeline = QueryFilterPipeline(DATASET_ROOT)
print(f"Remaining rows: {len(pipeline.filtered_df)}")

# Remove explicit document-context dependencies; cross-sentence anaphora is handled later.
pipeline.filter_self_contained_queries()
# pipeline.filter_no_summary_starts()
pipeline.filter_max_strangeness(0.3)
print(f"Remaining rows {len(pipeline.filtered_df)} after base filters")

# Reference-count filters define the benchmark candidate pool.
pipeline.filter_min_hit_queries(1)
pipeline.filter_min_source_count(25)
print(f"Remaining rows {len(pipeline.filtered_df)} after hit and source count filters")

# Remove explicit spoilers and self-citations.
pipeline.filter_no_et_al_queries()
pipeline.filter_no_citation_spoilers()
pipeline.citation_spoiler_audit.to_csv(
    DATASET_ROOT / 'benchmark_queries_v2_spoiler_audit.csv',
    index=False,
)
pipeline.filter_no_self_citations()
print(f"Remaining rows {len(pipeline.filtered_df)} after spoiler and self-citation filters")

pipeline.cleanup_queries()
pipeline.filter_min_query_length(6)
pipeline.filter_max_query_length(100)
pipeline.assert_no_citation_spoilers()
print(f"Remaining rows {len(pipeline.filtered_df)} after cleanup and length filters")

df_queries = pipeline.filtered_df[[
    'query_raw',
    'query',
    'citation_forms',
    'citation_cleanup_action',
    'citation_spoiler_reasons',
    'positive_author_metadata_complete',
    'context',
    'source_pmcid',
    'section_index',
    'paragraph_index',
    'sentence_index',
    'hits_parsed',
    'n_sources',
]].rename(columns={'hits_parsed': 'hits'})


Remaining rows: 261420
Remaining rows 249812 after base filters
Remaining rows 138579 after hit and source count filters
Remaining rows 108827 after spoiler and self-citation filters
Remaining rows 108357 after cleanup and length filters


## 4. Build the hard-negative benchmark

Select queries from source articles with one or two eligible queries. For every query, choose references cited elsewhere in the same article but not in the query sentence.


In [9]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('1 <= count <= 5')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

(1122, 5811)

In [11]:
selected_df = selected_df[[
    'source_pmcid',
    'section_index',
    'paragraph_index',
    'sentence_index',
    'query_raw',
    'context',
    'query',
    'citation_forms',
    'citation_cleanup_action',
    'citation_spoiler_reasons',
    'positive_author_metadata_complete',
    'hits_parsed',
    'selected_decoys',
]]
assert_no_residual_citation_spoilers(selected_df)
selected_df.to_csv(DATASET_ROOT / "benchmark_queries.csv", index=False)

## 5. Collect Abstracts and MeSH data

Fetch PubMed abstracts and MeSH annotations for every positive and hard-negative PMID. Requests are processed in batches and appended to local CSV stores.


In [14]:
import tqdm
import pandas as pd
from pathlib import Path
from pmcortex import PubMedMeSHClient, extract_abstracts_from_pubmed_xml, extract_mesh_terms_from_pubmed_xml

ABSTRACT_DB = DATASET_ROOT / "pmid_abstracts.csv"
MESH_DB = DATASET_ROOT / "pmid_mesh_terms.csv"
ABSTRACT_DB.unlink(missing_ok=True)
MESH_DB.unlink(missing_ok=True)

selected_df = pd.read_csv(DATASET_ROOT / "benchmark_queries.csv")
required_pmids = (
    set(selected_df['selected_decoys'].apply(eval).explode())
    .union(set(selected_df['hits_parsed'].apply(eval).explode()))
)


with PubMedMeSHClient() as client:
    for i in tqdm.tqdm(range(0, len(required_pmids), 1000), desc="Fetching PubMed metadata"):
        batch_pmids = list(required_pmids)[i:i+1000]
        xml = client.fetch_pubmed_xml(batch_pmids)
        abstracts = extract_abstracts_from_pubmed_xml(xml)
        mesh_terms = extract_mesh_terms_from_pubmed_xml(xml)
        
        abstracts.to_csv(ABSTRACT_DB, mode='a', header=not ABSTRACT_DB.exists(), index=False)
        mesh_terms.to_csv(MESH_DB, mode='a', header=not MESH_DB.exists(), index=False)

Fetching PubMed metadata: 100%|██████████| 7/7 [00:43<00:00,  6.16s/it]


## 6. Remove cross-context coreferences

Use FastCoref on `context + query` and remove a query only when an anaphoric mention in the query resolves to a mention in the preceding context. This is more selective than rejecting every query that merely contains a pronoun.

The language model must be installed explicitly. This notebook never downloads a model as a hidden side effect.


In [15]:
import spacy
from fastcoref import spacy_component  # Registers the FastCoref spaCy component.

from pmcortex import CrossContextCoreferenceFilter

SPACY_MODEL = "en_core_web_sm"
try:
    coref_nlp = spacy.load(SPACY_MODEL)
except OSError as error:
    raise RuntimeError(
        f"Install the spaCy model first: python -m spacy download {SPACY_MODEL}"
    ) from error

if "fastcoref" not in coref_nlp.pipe_names:
    coref_nlp.add_pipe("fastcoref")

cross_context_filter = CrossContextCoreferenceFilter(
    coref_nlp,
    batch_size=512,
)


08/20/2026 09:38:21 - INFO - 	 missing_keys: []
08/20/2026 09:38:21 - INFO - 	 unexpected_keys: []
08/20/2026 09:38:21 - INFO - 	 mismatched_keys: []
08/20/2026 09:38:21 - INFO - 	 error_msgs: []
08/20/2026 09:38:21 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [17]:
from pathlib import Path
import pandas as pd

BENCHMARK_DB = DATASET_ROOT / "benchmark_queries.csv"
COREF_FILTERED_DB = DATASET_ROOT / "benchmark_queries_coref_filtered.csv"

selected_df = pd.read_csv(BENCHMARK_DB)
rows_before_coreference = len(selected_df)
selected_df = cross_context_filter.filter_cross_context_coreferences(selected_df)
assert_no_residual_citation_spoilers(selected_df)
selected_df.to_csv(COREF_FILTERED_DB, index=False)

print(
    f"Removed {rows_before_coreference - len(selected_df)} cross-context queries; "
    f"{len(selected_df)} remain."
)


08/20/2026 09:38:39 - INFO - 	 Tokenize 512 inputs...
Map: 100%|██████████| 512/512 [00:00<00:00, 813.00 examples/s]
08/20/2026 09:38:40 - INFO - 	 ***** Running Inference on 512 texts *****
Inference: 100%|██████████| 512/512 [00:01<00:00, 433.06it/s]
08/20/2026 09:38:42 - INFO - 	 Tokenize 314 inputs...
Map: 100%|██████████| 314/314 [00:00<00:00, 847.73 examples/s]
08/20/2026 09:38:42 - INFO - 	 ***** Running Inference on 314 texts *****
Inference: 100%|██████████| 314/314 [00:00<00:00, 629.64it/s]

Removed 88 cross-context queries; 1034 remain.


## 7. Compare hard and soft negatives with MeSH

Load the coreference-filtered benchmark, retain same-article references as `hard_decoys`, and sample `soft_decoys` exclusively from other source articles. Both decoy lists receive positionally aligned descriptor-level MeSH Jaccard similarities. A missing MeSH annotation is represented by `None`, not by a misleading zero similarity.


In [ ]:
from ast import literal_eval
from collections.abc import Mapping
from random import Random

import pandas as pd
import tqdm

MESH_DB = DATASET_ROOT / "pmid_mesh_terms.csv"
QUERY_DB = DATASET_ROOT / "benchmark_queries_coref_filtered.csv"


def _parse_pmid_list(value: object) -> list[str]:
    """Parse a CSV cell containing a Python-style PMID list."""
    if isinstance(value, (list, tuple, set)):
        parsed_values = list(value)
    elif pd.isna(value):
        return []
    else:
        parsed = literal_eval(str(value))
        parsed_values = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]

    return [str(pmid).strip() for pmid in parsed_values if str(pmid).strip()]


all_mesh = pd.read_csv(MESH_DB, dtype={'pmid': 'string'})
selected_df = pd.read_csv(QUERY_DB, dtype={'source_pmcid': 'string'})
selected_df = selected_df.rename(columns={'selected_decoys': 'hard_decoys'})
selected_df['hard_decoys'] = selected_df['hard_decoys'].map(_parse_pmid_list)

positive_pmids = selected_df['hits_parsed'].map(_parse_pmid_list)
if not positive_pmids.map(len).eq(1).all():
    raise ValueError('Every selected query must contain exactly one positive PMID.')
selected_df['positive_pmid'] = positive_pmids.map(lambda pmids: pmids[0])

In [ ]:
SOFT_DECOYS_PER_QUERY = 80
RANDOM_SEED = 42


def _mesh_jaccard_similarity(
    positive_pmid: str,
    decoy_pmid: str,
    mesh_terms_by_pmid: Mapping[str, frozenset[str]],
) -> float | None:
    """Return descriptor-level MeSH Jaccard similarity, or None if terms are missing."""
    positive_terms = mesh_terms_by_pmid.get(positive_pmid, frozenset())
    decoy_terms = mesh_terms_by_pmid.get(decoy_pmid, frozenset())
    if not positive_terms or not decoy_terms:
        return None

    shared_term_count = len(positive_terms.intersection(decoy_terms))
    combined_term_count = len(positive_terms.union(decoy_terms))
    return shared_term_count / combined_term_count


mesh_terms_by_pmid = {
    str(pmid): frozenset(group['descriptor_name'].dropna().astype(str))
    for pmid, group in all_mesh.groupby('pmid', sort=False)
}
hard_decoys_by_source = {
    str(source_pmcid): frozenset(
        pmid
        for decoy_list in group['hard_decoys']
        for pmid in decoy_list
    )
    for source_pmcid, group in selected_df.groupby('source_pmcid', sort=False)
}
all_hard_decoy_pmids = frozenset(
    pmid
    for source_decoys in hard_decoys_by_source.values()
    for pmid in source_decoys
)

random_generator = Random(RANDOM_SEED)
soft_decoy_rows: list[list[str]] = []
hard_similarity_rows: list[list[float | None]] = []
soft_similarity_rows: list[list[float | None]] = []
positive_mesh_term_counts: list[int] = []

for row in tqdm.tqdm(
    selected_df.itertuples(index=False),
    total=len(selected_df),
    desc='Selecting decoys and calculating MeSH similarities',
):
    source_pmcid = str(row.source_pmcid)
    positive_pmid = str(row.positive_pmid)
    hard_decoys = list(row.hard_decoys)
    excluded_pmids = set(hard_decoys)
    excluded_pmids.add(positive_pmid)

    soft_pool = sorted(
        all_hard_decoy_pmids.difference(
            hard_decoys_by_source[source_pmcid],
            excluded_pmids,
        )
    )
    soft_pool_with_mesh = [pmid for pmid in soft_pool if mesh_terms_by_pmid.get(pmid)]
    soft_pool_without_mesh = [pmid for pmid in soft_pool if not mesh_terms_by_pmid.get(pmid)]

    with_mesh_count = min(SOFT_DECOYS_PER_QUERY, len(soft_pool_with_mesh))
    soft_decoys = random_generator.sample(soft_pool_with_mesh, with_mesh_count)
    missing_decoy_count = SOFT_DECOYS_PER_QUERY - len(soft_decoys)
    without_mesh_count = min(missing_decoy_count, len(soft_pool_without_mesh))
    soft_decoys.extend(random_generator.sample(soft_pool_without_mesh, without_mesh_count))

    hard_similarities = [
        _mesh_jaccard_similarity(positive_pmid, decoy, mesh_terms_by_pmid)
        for decoy in hard_decoys
    ]
    soft_similarities = [
        _mesh_jaccard_similarity(positive_pmid, decoy, mesh_terms_by_pmid)
        for decoy in soft_decoys
    ]

    soft_decoy_rows.append(soft_decoys)
    hard_similarity_rows.append(hard_similarities)
    soft_similarity_rows.append(soft_similarities)
    positive_mesh_term_counts.append(len(mesh_terms_by_pmid.get(positive_pmid, frozenset())))

selected_df['soft_decoys'] = soft_decoy_rows
selected_df['hard_decoy_mesh_similarities'] = hard_similarity_rows
selected_df['soft_decoy_mesh_similarities'] = soft_similarity_rows
selected_df['n_positive_mesh_terms'] = positive_mesh_term_counts

In [ ]:
hard_lengths_match = (
    selected_df['hard_decoys'].map(len)
    == selected_df['hard_decoy_mesh_similarities'].map(len)
).all()
soft_lengths_match = (
    selected_df['soft_decoys'].map(len)
    == selected_df['soft_decoy_mesh_similarities'].map(len)
).all()
if not hard_lengths_match or not soft_lengths_match:
    raise RuntimeError('Every decoy must have one aligned MeSH-similarity value.')

mesh_similarity_coverage = pd.Series(
    {
        'hard_decoys': selected_df['hard_decoy_mesh_similarities'].explode().notna().mean(),
        'soft_decoys': selected_df['soft_decoy_mesh_similarities'].explode().notna().mean(),
    },
    name='mesh_similarity_coverage',
).to_frame()
display(mesh_similarity_coverage)

assert_no_residual_citation_spoilers(selected_df)
selected_df.to_csv(DATASET_ROOT / "benchmark_queries_v2_softdecoys.csv", index=False)
display(
    selected_df[[
        'positive_pmid',
        'hard_decoys',
        'hard_decoy_mesh_similarities',
        'soft_decoys',
        'soft_decoy_mesh_similarities',
    ]].head()
)

In [ ]:
selected_df.drop(columns=['citation_forms', 'citation_cleanup_action', 'citation_spoiler_reasons', 'positive_author_metadata_complete'], inplace=True)

In [23]:
selected_df

,source_pmcid,section_index,paragraph_index,sentence_index,query_raw,context,query,hits_parsed,selected_decoys
0,PMC10047863,0,0,0,"Definitive, as well as post-operative, radioth...",NaN,"Definitive, as well as post-operative, radioth...",29709262,"[12074474, 12945975, 16256277, 17576450, 19801..."
1,PMC10047863,0,0,2,"However, during the long course of radiotherap...",Recent advances in radiation therapy technolog...,"During the long course of radiotherapy, patien...",30076479,"[12074474, 12945975, 16256277, 17576450, 19801..."
2,PMC10047863,3,4,0,Adaptive radiotherapy seems to be able to comp...,"In the study conducted by Jensen et al., the m...",Adaptive radiotherapy seems to be able to comp...,25573091,"[12074474, 12945975, 16256277, 17576450, 19801..."
3,PMC10093920,5,4,6,The BALB/c recipients used here are albinos an...,"Again, macrophage infiltration of gmRPC grafts...",The BALB/c recipients used here are albinos an...,25895728,"[12832694, 17565557, 24527394, 25604685, 28962..."
4,PMC10093920,5,5,5,"Importantly, the degree of macrophage infiltra...","Moreover, sustained survival has been seen rep...",The degree of macrophage infiltration seen in ...,34172826,"[12832694, 17565557, 24527394, 25604685, 28962..."
...,...,...,...,...,...,...,...,...,...
1029,PMC9939934,0,1,2,The first endocrine organoid-based disease mod...,"These advances span the generation of breast, ...",The first endocrine organoid-based disease mod...,32771978,"[27308531, 33730555, 33692550, 27590160, 27494..."
1030,PMC9978987,0,0,1,Biopsy is generally performed on the anterior ...,NaN,Biopsy is generally performed on the anterior ...,12967736,"[11577117, 15815930, 19473858, 19455356, 10929..."
1031,PMC9978987,0,0,3,Pain is also influenced by the emotional statu...,"A long procedure duration (>30 minutes), limit...",Pain is also influenced by the emotional statu...,21287348,"[11577117, 15815930, 19473858, 19455356, 10929..."
1032,PMC9978987,0,0,10,A more recent study was conducted with patient...,"Digital therapeutics have recently emerged, an...",A more recent study was conducted with patient...,36222814,"[11577117, 15815930, 19473858, 19455356, 10929..."
